> **中文阅读版**：本文件是 `04_alpaca_paper_trading_demo.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# Alpaca 模拟交易演示

**Docker 镜像**：`ml4t`

**书中参考**：第 25 章，第 25.3 节（与 Alpaca 集成）

[`03_ib_paper_trading_demo`](03_ib_paper_trading_demo.ipynb) 将同一个策略连接到了
Interactive Brokers。本 notebook 将其连接到 Alpaca，而有趣之处在于改动如此之少：不同的 broker 类、不同的 feed 类，相同的 `Strategy`、相同的 shadow 模式、相同的订单日志。两个 broker 的区别在于接入成本，而不在于策略必须对它们了解什么。

默认情况下，它甚至完全不需要 Alpaca 账户即可运行。离线路径在确定性 bar 上使用模拟
broker，这正是该 notebook 能在持续集成中执行、并且任何尚未注册任何服务的人都能读懂的原因。该路径所展示的关于策略接口的一切都是真实的；而它本会展示的关于 Alpaca 的一切则不是真实的，并且 notebook 在每一步都明确说明哪个是哪个。

**学习目标**
- 在允许策略接近 broker 之前，检查环境、SDK 和账户状态，并将每一项视为独立的闸门
- 将策略包装为 shadow 模式，使订单被记录而绝不会发送
- 当 broker 是模拟的时，分清一次运行真正确立了什么、又只是走过了什么流程

**前置条件**：[`01_unified_framework_demo`](01_unified_framework_demo.ipynb) 中共享的
`Strategy` 接口。Alpaca 模拟账户以及 `ALPACA_API_KEY` / `ALPACA_SECRET_KEY` 仅实时路径需要，而该路径默认关闭。

In [1]:
"""Connect ml4t strategies to Alpaca with shadow-mode risk controls."""

import asyncio
import logging
import os
import warnings
from datetime import UTC, datetime, timedelta

import numpy as np
import polars as pl
from async_utils import run_async
from ml4t.backtest import OrderSide, Strategy

from utils.paths import display_path, get_output_dir
from utils.reproducibility import set_global_seeds

# The broker adapters pull in websockets' legacy module, which deprecates itself on import, so
# the filter has to be in force before the import rather than after it. It is the library's
# business rather than this notebook's and nothing in the result depends on it.
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"websockets\.legacy")

# alpaca-py is an optional broker SDK; the simulated path runs without it. The
# try/except is the only such optional import in the notebook.
HAS_ALPACA_SDK = False
try:
    import alpaca  # noqa: F401
    from alpaca.trading.client import TradingClient
    from ml4t.live import AlpacaBroker, AlpacaDataFeed, LiveEngine, LiveRiskConfig
    from ml4t.live.safety import SafeBroker

    HAS_ALPACA_SDK = True
except ImportError:
    pass

# Give this notebook its own logger rather than configuring the root one: basicConfig is a no-op
# once an imported library has attached a handler, and reconfiguring the root would either do
# nothing or print every line twice.
logger = logging.getLogger("alpaca_paper_demo")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    _handler = logging.StreamHandler()
    _handler.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))
    logger.addHandler(_handler)
logging.getLogger("alpaca").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

if HAS_ALPACA_SDK:
    print("[OK] ml4t.live Alpaca components imported")
else:
    print("Alpaca SDK not installed (uv add alpaca-py); running simulation only")

[OK] ml4t.live Alpaca components imported


In [2]:
DEMO_DURATION_SECONDS = 60
MAX_SYMBOLS = 0
SIMULATION_STEPS = 10
LIVE_FEED = 0  # explicit opt-in; default execution is offline and paper-safe
SEED = 42

首先是一个环境变量覆盖。Alpaca 的 WebSocket 循环与无头 notebook runner 所安装的嵌套事件循环
并不兼容：`asyncio.wait_for` 无法可靠地取消内部的流式任务，因此无人值守的运行会超过其自身的时长限制而不是结束。当 runner 表明自身身份时，实时 feed 会被关闭，模拟路径会运行。交互式 Jupyter 会话不受影响。

在无法自我监督的环境中强制走安全路径是正确的默认做法，而运行会在输出中说明这一点，而不是静默地走另一条分支。

In [3]:
if os.environ.get("ML4T_HEADLESS_PAPERMILL") == "1":
    LIVE_FEED = 0
    print("Headless papermill detected: LIVE_FEED disabled, simulated path will run")

set_global_seeds(SEED)

ALPACA_API_KEY = os.environ.get("ALPACA_API_KEY", "")
ALPACA_SECRET_KEY = os.environ.get("ALPACA_SECRET_KEY", "")
PAPER_TRADING = True

ALL_SYMBOLS = ["SPY", "QQQ", "IWM"]
SYMBOLS = ALL_SYMBOLS[:MAX_SYMBOLS] if MAX_SYMBOLS > 0 else ALL_SYMBOLS.copy()

## 1. 凭证验证

Alpaca 需要 API key 进行身份验证。对于模拟交易：
- 在 https://alpaca.markets 注册
- 在仪表盘中生成 API key
- 设置环境变量（绝不要硬编码！）

In [4]:
def verify_credentials():
    """Check that API credentials and SDK are available."""
    if not HAS_ALPACA_SDK:
        print("\n" + "=" * 60)
        print("ALPACA SDK NOT INSTALLED")
        print("=" * 60)
        print("\nTo connect to Alpaca, install the SDK:")
        print("   uv add alpaca-py")
        print("\nRunning in DEMO MODE with simulated broker...")
        return False

    if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
        print("\n" + "=" * 60)
        print("ALPACA CREDENTIALS NOT FOUND")
        print("=" * 60)
        print("\nTo run this notebook with real Alpaca connection:")
        print("1. Create an Alpaca account at https://alpaca.markets")
        print("2. Generate API keys in the dashboard")
        print("3. Set environment variables:")
        print("   export ALPACA_API_KEY='PKXXXXXXXX'")
        print("   export ALPACA_SECRET_KEY='xxxxxxxxxx'")
        print("\nRunning in DEMO MODE with simulated broker...")
        return False

    print("\n" + "=" * 60)
    print("ALPACA CREDENTIALS VERIFIED")
    print("=" * 60)
    print(f"Paper Trading: {'YES' if PAPER_TRADING else 'NO (LIVE!)'}")
    return True


HAS_CREDENTIALS = verify_credentials()


ALPACA CREDENTIALS NOT FOUND

To run this notebook with real Alpaca connection:
1. Create an Alpaca account at https://alpaca.markets
2. Generate API keys in the dashboard
3. Set environment variables:
   export ALPACA_API_KEY='PKXXXXXXXX'
   export ALPACA_SECRET_KEY='xxxxxxxxxx'

Running in DEMO MODE with simulated broker...


**发现**：凭证闸门明确区分缺少 SDK 与缺少密钥，这使得执行模式在 notebook 接触 broker 连接之前就可观察。

**交易含义**：实时 notebook 绝不应隐藏自己是已认证、shadow 模式还是完全模拟，因为该状态决定了每个下游操作的操作风险。

## 2. 连接 Alpaca

AlpacaBroker 类负责：
- 用于订单和账户信息的 REST API
- 用于实时订单更新的 WebSocket
- 自动持仓/订单同步

In [5]:
def get_alpaca_account_snapshot():
    """Fetch an account snapshot without starting the streaming broker session."""
    if not HAS_CREDENTIALS:
        return None, None, None, None

    print("\n" + "=" * 60)
    print("CONNECTING TO ALPACA")
    print("=" * 60)

    trading_client = TradingClient(
        api_key=ALPACA_API_KEY,
        secret_key=ALPACA_SECRET_KEY,
        paper=PAPER_TRADING,
    )
    account = trading_client.get_account()
    raw_positions = trading_client.get_all_positions()

    broker = AlpacaBroker(
        api_key=ALPACA_API_KEY,
        secret_key=ALPACA_SECRET_KEY,
        paper=PAPER_TRADING,
    )

    print("\nConnected to Alpaca")
    print(f"   Paper Trading: {'YES' if PAPER_TRADING else 'NO'}")

    nlv = float(account.equity)
    cash = float(account.cash)

    print("\nACCOUNT READINESS")
    print("   Account values received: [OK]")

    # Get positions
    positions = {
        position.symbol: {
            "quantity": float(position.qty),
            "entry_price": float(position.avg_entry_price),
            "current_price": float(position.current_price or position.avg_entry_price),
        }
        for position in raw_positions
    }
    print(f"   Open positions received: {len(positions)}")

    return broker, nlv, cash, positions


# Run connection
if LIVE_FEED and HAS_CREDENTIALS and HAS_ALPACA_SDK:
    try:
        broker, nlv, cash, positions = get_alpaca_account_snapshot()
    except Exception as e:
        print(f"\nConnection failed: {e}")
        print("\nTroubleshooting:")
        print("1. Are your API keys correct?")
        print("2. Is your account enabled for paper trading?")
        print("3. Check https://status.alpaca.markets for outages")
        raise RuntimeError("Alpaca paper session unavailable") from e
else:
    broker = None
    print("Offline mode selected; no Alpaca account request was made.")

Offline mode selected; no Alpaca account request was made.


**发现**：账户摘要确认，broker 适配器暴露的现金、权益和持仓状态，与策略稍后在会话中将依赖的状态相同。

**交易含义**：实时策略应始终在开始监听市场数据之前证明其 broker 快照是正常的；否则即使是正确的信号，也可能带着陈旧持仓被路由。

## 3. ETF 动量策略

该策略与我们在回测中使用的策略**完全相同**。
使用包含 SPY、QQQ、IWM 的 ETF 案例研究。

In [6]:
class ETFMomentumStrategy(Strategy):
    """ETF momentum strategy for live trading.

    Tracks 5-day momentum across ETFs and generates signals
    when momentum crosses thresholds.

    This is the SAME code used in backtest - zero changes for live!
    """

    def __init__(self, lookback: int = 5, threshold: float = 0.02, position_size: int = 10):
        self.lookback = lookback
        self.threshold = threshold
        self.position_size = position_size
        self.prices: dict[str, list[float]] = {}
        self.signals: list[dict] = []

    def on_start(self, broker):
        """Called when engine starts."""
        logger.info(f"Strategy started: ETFMomentum(lookback={self.lookback})")
        for symbol in SYMBOLS:
            self.prices[symbol] = []

    def on_data(self, timestamp: datetime, data: dict, context: dict, broker):
        """Called for each bar.

        Args:
            timestamp: Bar timestamp
            data: {symbol: {'open', 'high', 'low', 'close', 'volume'}}
            context: Additional metadata (vwap, trade_count, etc.)
            broker: Broker instance (sync interface)
        """
        for symbol, bar in data.items():
            if symbol not in self.prices:
                self.prices[symbol] = []

            close = bar["close"]
            self.prices[symbol].append(close)

            # Calculate momentum
            if len(self.prices[symbol]) > self.lookback:
                old_price = self.prices[symbol][-self.lookback - 1]
                momentum = (close - old_price) / old_price

                # Get current position
                position = broker.get_position(symbol)
                has_position = position is not None and position.quantity > 0

                # Generate signals
                if momentum > self.threshold and not has_position:
                    signal = {
                        "timestamp": timestamp,
                        "symbol": symbol,
                        "action": "BUY",
                        "momentum": momentum,
                        "price": close,
                    }
                    self.signals.append(signal)
                    logger.info(f"BUY {symbol}: Momentum {momentum:.2%} > {self.threshold:.2%}")
                    broker.submit_order(symbol, self.position_size, side=OrderSide.BUY)

                elif momentum < -self.threshold and has_position:
                    signal = {
                        "timestamp": timestamp,
                        "symbol": symbol,
                        "action": "SELL",
                        "momentum": momentum,
                        "price": close,
                    }
                    self.signals.append(signal)
                    logger.info(f"SELL {symbol}: Momentum {momentum:.2%} < -{self.threshold:.2%}")
                    broker.submit_order(symbol, self.position_size, side=OrderSide.SELL)

    def on_end(self, broker):
        """Called when engine stops."""
        logger.info(f"Strategy ended. Signals generated: {len(self.signals)}")

### 结构化信号与订单日志

实时路径和模拟路径都会将信号与订单意图累积到一个 Polars frame 中，因此审查者可以检查策略在每一步做出的决定，且无论 broker 是 Alpaca 还是 notebook 内的 mock，其结构都相同。

In [7]:
def signals_to_frame(signals: list[dict]) -> pl.DataFrame:
    """Render the strategy's signal accumulator as a Polars frame for display."""
    if not signals:
        return pl.DataFrame(
            schema={
                "timestamp": pl.Datetime,
                "symbol": pl.String,
                "action": pl.String,
                "momentum": pl.Float64,
                "price": pl.Float64,
            }
        )
    return pl.DataFrame(signals)

## 4. 安全 broker 配置

在实盘之前，我们用 `SafeBroker` 包装 broker，它提供：
- Shadow 模式（仅虚拟订单）
- 持仓限制
- 订单速率限制
- 紧急停止开关

In [8]:
def create_safe_broker(underlying_broker):
    """Create SafeBroker with risk configuration."""
    risk_state_path = get_output_dir(25, "alpaca_paper_demo") / "risk_state.json"
    risk_config = LiveRiskConfig(
        shadow_mode=True,  # CRITICAL: Virtual orders only!
        max_position_value=50_000.0,
        max_order_value=10_000.0,
        max_orders_per_minute=10,
        dedup_window_seconds=0.0,  # Disable for demo
        state_file=str(risk_state_path),
    )

    safe_broker = SafeBroker(underlying_broker, risk_config)

    print("\n" + "=" * 60)
    print("RISK CONFIGURATION (SHADOW MODE)")
    print("=" * 60)
    print("   Shadow Mode: ENABLED (no real orders)")
    print(f"   Max Position Value: ${risk_config.max_position_value:,.0f}")
    print(f"   Max Order Value: ${risk_config.max_order_value:,.0f}")
    print(f"   Rate Limit: {risk_config.max_orders_per_minute}/minute")
    print(f"   Risk State: {display_path(risk_state_path)}")

    return safe_broker, risk_config

**发现**：风险配置的打印输出使 shadow 模式和敞口限制在实时 feed 开始发送数据之前就可见。

**交易含义**：Broker 包装器应显式暴露其生效中的限制，因为实时上线的安全性只取决于运行时实际启用的控制措施。

## 5. 实时数据 Feed

AlpacaDataFeed 订阅实时市场数据：
- **bars**：OHLCV 聚合（默认 1 分钟）
- **quotes**：带有数量的买价/卖价
- **trades**：逐笔成交

数据源：
- **IEX**：免费，对部分 symbol 有 15 分钟延迟
- **SIP**：付费高级版，来自所有交易所的实时数据

### 模拟路径：扁平字典投资组合

当凭证或 SDK 缺失时，演示会针对 notebook 内的一个小型 broker 运行。投资组合是一个扁平字典（`{"cash": float, "positions": {symbol: {qty, entry_price}}}`），而不是嵌套 dataclass，因此模拟状态可以直接读入 Polars frame，以便与实时路径并排显示。

In [9]:
class MockBroker:
    """Minimal sync broker for the no-credential simulation path.

    The portfolio is a flat dict to keep the simulation state inspectable; it is
    intentionally not a SafeBroker wrapper because shadow-mode requires fresh
    quotes that the simulation does not produce.
    """

    REF_PRICES = {"SPY": 600.0, "QQQ": 520.0, "IWM": 225.0}

    def __init__(self, initial_cash: float = 100_000.0):
        self.portfolio = {"cash": initial_cash, "positions": {}}
        self.order_log: list[dict] = []
        self.current_prices = dict(self.REF_PRICES)
        self.current_timestamp = datetime(2025, 1, 2, tzinfo=UTC)

    def update_market(self, timestamp: datetime, prices: dict[str, float]) -> None:
        """Update the simulated market used for subsequent fills."""
        self.current_timestamp = timestamp
        self.current_prices.update(prices)

    def get_position(self, symbol: str):
        pos = self.portfolio["positions"].get(symbol)
        if pos is None:
            return None

        class _PosView:
            quantity = pos["quantity"]

        return _PosView()

    def submit_order(self, asset: str, quantity: int, side=None, **kwargs) -> dict:
        price = self.current_prices[asset]
        if isinstance(side, OrderSide):
            side_name = side.value.upper()
        elif side is None:
            side_name = "BUY"
        else:
            side_name = str(getattr(side, "value", side)).upper()
        status = "rejected"
        if side is None or side == OrderSide.BUY:
            cost = quantity * price
            if cost <= self.portfolio["cash"]:
                self.portfolio["cash"] -= cost
                pos = self.portfolio["positions"].get(asset)
                if pos is None:
                    self.portfolio["positions"][asset] = {
                        "quantity": quantity,
                        "entry_price": price,
                    }
                else:
                    total_qty = pos["quantity"] + quantity
                    avg = (pos["quantity"] * pos["entry_price"] + quantity * price) / total_qty
                    self.portfolio["positions"][asset] = {
                        "quantity": total_qty,
                        "entry_price": avg,
                    }
                status = "filled"
        elif side == OrderSide.SELL:
            pos = self.portfolio["positions"].get(asset)
            if pos is not None and pos["quantity"] >= quantity:
                self.portfolio["cash"] += quantity * price
                remaining = pos["quantity"] - quantity
                if remaining == 0:
                    del self.portfolio["positions"][asset]
                else:
                    self.portfolio["positions"][asset] = {
                        "quantity": remaining,
                        "entry_price": pos["entry_price"],
                    }
                status = "filled"
        else:
            status = "unsupported"
        order = {
            "order_id": f"SIM-{len(self.order_log) + 1}",
            "timestamp": self.current_timestamp,
            "symbol": asset,
            "side": side_name,
            "quantity": quantity,
            "price": price,
            "status": status,
        }
        self.order_log.append(order)
        return order

### 实盘路径：引擎装配

实盘路径构建 `AlpacaDataFeed > SafeBroker > LiveEngine`。每个辅助函数都小到可以一眼检查；`run_engine_for_duration` 是唯一的异步部分，而 `display_engine_results` 是纯只读后处理。

In [10]:
def create_alpaca_engine(strategy):
    """Build the AlpacaDataFeed, SafeBroker, and LiveEngine wiring."""
    safe_broker, _ = create_safe_broker(broker)
    feed = AlpacaDataFeed(
        api_key=ALPACA_API_KEY,
        secret_key=ALPACA_SECRET_KEY,
        symbols=SYMBOLS,
        data_type="bars",
        feed="iex",
    )
    engine = LiveEngine(strategy=strategy, broker=safe_broker, feed=feed)
    # Alpaca SDK retries aggressively under nest_asyncio; quiet the retry logs.
    for name in [
        "alpaca",
        "alpaca.data",
        "alpaca.data.live",
        "alpaca.data.live.websocket",
        "alpaca.trading.stream",
        "websockets",
    ]:
        logging.getLogger(name).setLevel(logging.CRITICAL)
    return engine, safe_broker, feed

In [11]:
async def run_engine_for_duration(engine, duration_s: int):
    """Connect the engine and let it stream for at most `duration_s` seconds."""
    await asyncio.wait_for(engine.connect(), timeout=10)
    try:
        await asyncio.wait_for(engine.run(), timeout=duration_s)
    except TimeoutError:
        print(f"Demo duration ({duration_s}s) reached")

In [12]:
def display_engine_results(strategy, safe_broker, feed, engine):
    """Print engine stats and render the strategy's signal log as a Polars frame."""
    print("Engine stats:", {k: engine.stats[k] for k in list(engine.stats)[:6]})
    print("Feed stats: ", {k: feed.stats[k] for k in list(feed.stats)[:6]})

    vp = safe_broker._virtual_portfolio
    print(f"Virtual Portfolio cash: ${vp.cash:,.2f}")
    for symbol, pos in vp.positions.items():
        value = pos.quantity * (pos.current_price or pos.entry_price)
        print(f"   {symbol}: {pos.quantity} shares @ ${pos.entry_price:.2f} = ${value:,.2f}")

    print(f"\nSignals: {len(strategy.signals)}")
    return signals_to_frame(strategy.signals)

In [13]:
async def run_live_demo_with_feed():
    """Run the explicitly selected Alpaca or offline simulation path."""
    if not LIVE_FEED:
        print(f"LIVE_FEED={LIVE_FEED}: running the offline simulation")
        return await run_simulated_demo()
    if not HAS_CREDENTIALS or broker is None:
        raise RuntimeError("LIVE_FEED requires the Alpaca SDK and paper credentials")

    print("LIVE TRADING DEMO (Shadow Mode)")
    strategy = ETFMomentumStrategy(lookback=5, threshold=0.02, position_size=10)
    engine, safe_broker, feed = create_alpaca_engine(strategy)
    print(f"Starting live engine for {DEMO_DURATION_SECONDS}s; watching {', '.join(SYMBOLS)}")

    try:
        await run_engine_for_duration(engine, DEMO_DURATION_SECONDS)
    finally:
        feed.stop()
    return display_engine_results(strategy, safe_broker, feed, engine), pl.DataFrame()

In [14]:
async def run_simulated_demo() -> tuple[pl.DataFrame, pl.DataFrame]:
    """Run the strategy against the flat-dict MockBroker; return the signal log."""
    print("SIMULATED DEMO (no live Alpaca feed)")
    strategy = ETFMomentumStrategy(lookback=3, threshold=0.01, position_size=10)
    mock_broker = MockBroker()
    strategy.on_start(mock_broker)

    set_global_seeds(SEED)
    base_prices = dict(MockBroker.REF_PRICES)
    start = datetime(2025, 1, 2, 14, 30, tzinfo=UTC)
    for step in range(SIMULATION_STEPS):
        timestamp = start + timedelta(minutes=step)
        data = {}
        for symbol in SYMBOLS:
            base_prices[symbol] *= 1 + np.random.normal(0.001, 0.01)
            data[symbol] = {
                "open": base_prices[symbol] * 0.999,
                "high": base_prices[symbol] * 1.002,
                "low": base_prices[symbol] * 0.998,
                "close": base_prices[symbol],
                "volume": int(np.random.randint(100000, 1000000)),
            }
        mock_broker.update_market(timestamp, {symbol: bar["close"] for symbol, bar in data.items()})
        strategy.on_data(timestamp, data, {}, mock_broker)
    strategy.on_end(mock_broker)

    print(f"Simulated cash: ${mock_broker.portfolio['cash']:,.2f}")
    for symbol, pos in mock_broker.portfolio["positions"].items():
        value = pos["quantity"] * pos["entry_price"]
        print(f"   {symbol}: {pos['quantity']} shares @ ${pos['entry_price']:.2f} = ${value:,.2f}")
    print(f"Signals: {len(strategy.signals)}; Orders: {len(mock_broker.order_log)}")

    signal_frame = signals_to_frame(strategy.signals)
    order_frame = (
        pl.DataFrame(mock_broker.order_log)
        if mock_broker.order_log
        else pl.DataFrame(
            schema={
                "order_id": pl.String,
                "timestamp": pl.Datetime,
                "symbol": pl.String,
                "side": pl.String,
                "quantity": pl.Int64,
                "price": pl.Float64,
                "status": pl.String,
            }
        )
    )
    return signal_frame, order_frame

In [15]:
# Run the demo
demo_signal_log, demo_order_log = run_async(run_live_demo_with_feed())
demo_signal_log

2026-09-08 20:52:24,652 - alpaca_paper_demo - INFO - Strategy started: ETFMomentum(lookback=3)


2026-09-08 20:52:24,654 - alpaca_paper_demo - INFO - BUY QQQ: Momentum 1.15% > 1.00%


2026-09-08 20:52:24,654 - alpaca_paper_demo - INFO - BUY IWM: Momentum 1.76% > 1.00%


2026-09-08 20:52:24,654 - alpaca_paper_demo - INFO - Strategy ended. Signals generated: 2


LIVE_FEED=0: running the offline simulation
SIMULATED DEMO (no live Alpaca feed)
Simulated cash: $92,461.65
   QQQ: 10 shares @ $525.77 = $5,257.66
   IWM: 10 shares @ $228.07 = $2,280.69
Signals: 2; Orders: 2


timestamp,symbol,action,momentum,price
"datetime[μs, UTC]",str,str,f64,f64
2025-01-02 14:33:00 UTC,"""QQQ""","""BUY""",0.011475,525.765796
2025-01-02 14:35:00 UTC,"""IWM""","""BUY""",0.017552,228.069118


In [16]:
demo_order_log

order_id,timestamp,symbol,side,quantity,price,status
str,"datetime[μs, UTC]",str,str,i64,f64,str
"""SIM-1""",2025-01-02 14:33:00 UTC,"""QQQ""","""BUY""",10,525.765796,"""filled"""
"""SIM-2""",2025-01-02 14:35:00 UTC,"""IWM""","""BUY""",10,228.069118,"""filled"""


**发现**：所选执行模式是显式的。离线路径通过可检查的 broker 演练相同的策略接口，而选择性启用的实盘路径则增加 `SafeBroker` 和 Alpaca 传输控制。

**交易含义**：保持实盘路径和模拟路径在结构上对齐，更容易发现真正的 broker 侧问题，而不是调试由演示环境引入的差异。

## 6. 订单类型演示

Alpaca 支持多种订单类型：
- **MARKET**：以最优可用价格立即成交
- **LIMIT**：以指定价格或更优价格成交
- **STOP**：当价格触及止损价时触发市价单
- **STOP_LIMIT**：当价格触及止损价时触发限价单

In [17]:
async def demonstrate_order_types():
    """Demonstrate different order types (shadow mode)."""
    if not HAS_CREDENTIALS or broker is None:
        print("\nSkipping order demo - no credentials")
        return
    if not LIVE_FEED:
        # SafeBroker requires fresh market data for staleness checks; without the
        # live WebSocket feed there are no recent quotes, so submit_order_async
        # raises RiskLimitError. Skip the demo under headless papermill.
        print(f"\nLIVE_FEED={LIVE_FEED}: skipping order-type demo (needs live market data)")
        return

    print("\n" + "=" * 60)
    print("ORDER TYPE DEMONSTRATIONS (Shadow Mode)")
    print("=" * 60)

    safe_broker, _ = create_safe_broker(broker)

    from ml4t.backtest.types import OrderType

    # Market order
    print("\n1. MARKET ORDER")
    order = await safe_broker.submit_order_async("SPY", 10, side=OrderSide.BUY)
    print(f"   Order ID: {order.order_id}")
    print("   Type: MARKET")
    print(f"   Status: {order.status.value}")

    # Limit order
    print("\n2. LIMIT ORDER")
    order = await safe_broker.submit_order_async(
        "QQQ", 5, side=OrderSide.BUY, order_type=OrderType.LIMIT, limit_price=500.00
    )
    print(f"   Order ID: {order.order_id}")
    print("   Type: LIMIT @ $500.00")
    print(f"   Status: {order.status.value}")

    # Stop order
    print("\n3. STOP ORDER")
    order = await safe_broker.submit_order_async(
        "IWM", 10, side=OrderSide.SELL, order_type=OrderType.STOP, stop_price=220.00
    )
    print(f"   Order ID: {order.order_id}")
    print("   Type: STOP @ $220.00")
    print(f"   Status: {order.status.value}")

    # Show virtual portfolio
    vp = safe_broker._virtual_portfolio
    print("\nVirtual Portfolio After Orders:")
    print(f"   Cash: ${vp.cash:,.2f}")
    for symbol, pos in vp.positions.items():
        print(f"   {symbol}: {pos.quantity} shares")


run_async(demonstrate_order_types())


Skipping order demo - no credentials


**发现**：可选的实盘路径为每种订单类型定义了影子模式示例。默认离线运行会显式跳过这些提交，因为它没有 Alpaca 客户端、实时行情源或 `SafeBroker`。

**交易含义**：在凭证和实时模拟行情源可用之后，影子模式提交是中间阶段，可在不产生敞口的情况下测试路由、校验和防护规则。

## 7. 干净关闭

会话应以显式断开连接结束，这样下一次运行会从已知的 broker 状态开始，而不是继承过期的订阅或会话假设。

In [18]:
# Disconnect from Alpaca
if broker is not None:
    run_async(broker.disconnect())
    print("\nDisconnected from Alpaca")

## 总结

本 notebook 定义了完整的可选 Alpaca 集成工作流，并默认执行确定性的离线策略/broker 路径。在 `LIVE_FEED=True`、凭证和 SDK 可用时，它还可以演练：

1. **认证**：通过环境变量提供 API key/secret
2. **连接**：访问模拟交易账户
3. **账户信息**：查询权益、现金、持仓
4. **实时行情源**：订阅 bars/quotes/trades
5. **安全交易**：在影子模式下使用 SafeBroker
6. **订单类型**：市价单、限价单、止损单

### Alpaca 与 IB 对比

| 功能 | Alpaca | Interactive Brokers |
|---------|--------|---------------------|
| 最低余额 | 无 | 无（但保证金要求更高） |
| 佣金 | 免费 | 每笔交易 $0-1 |
| 实时数据 | 免费（IEX） | 付费订阅 |
| API 复杂度 | 简单 REST | 复杂的 socket 协议 |
| 加密货币 | 支持（24/7） | 有限 |
| 模拟交易 | 支持 | 支持 |

### 后续步骤

1. 以影子模式运行 1-2 周
2. 验证信号与回测预期一致
3. 启用模拟交易（`shadow_mode=False`）
4. 在模拟盘监控 2-4 周
5. 逐步以小额仓位过渡到实盘

### 加密货币交易

参见 `05_alpaca_crypto_live_demo.py`，了解 24/7 加密货币交易演示。

In [19]:
print("\n" + "=" * 60)
print("ALPACA PAPER TRADING DEMO COMPLETE")
print("=" * 60)
print(f"Symbols: {', '.join(SYMBOLS)}")
print(f"Paper Trading: {'YES' if PAPER_TRADING else 'NO'}")
shadow_mode_state = "ENABLED" if LIVE_FEED else "NOT ACTIVE (offline simulation)"
print(f"Shadow Mode: {shadow_mode_state}")
execution_mode = "Alpaca shadow feed" if LIVE_FEED else "offline simulation"
print(f"Execution Mode: {execution_mode}")
print("The same ETFMomentumStrategy interface drives the selected execution path.")


ALPACA PAPER TRADING DEMO COMPLETE
Symbols: SPY, QQQ, IWM
Paper Trading: YES
Shadow Mode: NOT ACTIVE (offline simulation)
Execution Mode: offline simulation
The same ETFMomentumStrategy interface drives the selected execution path.


## 关键要点

1. **broker 是变更中最小的一部分。** 从 Interactive Brokers 迁移到 Alpaca，只替换了一个 broker 类和一个行情源类。策略、风险包装器、订单日志和引擎仍是相同的对象。这正是 broker 抽象的意义所在，也是选择交易场所属于运维决策而非研究决策的原因。
2. **分别检查环境、SDK 和账户。** 它们的失败方式不同，修复方式也不同：缺少 key 是配置问题，缺少 package 是安装问题，账户被拒是权限问题。一个笼统的“连接失败”会把三者都掩盖掉。
3. **影子模式是实盘部署应起步的地方。** 真实价格、真实账户状态、真实策略，以及只记录不发送的订单。除了那一步不可撤销的操作之外，其他都是真实的。
4. **说明一次运行中哪些部分是真实的。** 这次运行是离线的，因此它只证明了策略接口和订单日志可以工作，完全没有证明 Alpaca 的成交、延迟或拒单行为。一个不区分两者的演示，会让读者去信任错误的那一半。

**此处所构建内容的已知局限。** 默认路径从不联系 Alpaca，因此所有真正重要的故障模式——拒单、部分成交、socket 断开、报价过期——都不会出现在其中。模拟 K 线来自固定随机种子，而不是来自市场。而且影子模式证明的是订单没有被发送，而不是证明它们会以假设的价格成交。

**下一步**：[`05_alpaca_crypto_live_demo`](05_alpaca_crypto_live_demo.ipynb) 将相同的部署模式应用到一个永不收盘的市场，从而移除了本 notebook 可以依赖的会话边界。